# 01 — Data Loading & Source Exploration

**Purpose:** Load all 9 raw source CSV files, inspect their shapes and column types, and document what each table contributes to the pipeline.  
**Output:** No files saved. This notebook is read-only — all outputs flow into `02_column_cleaning.ipynb`.

---

## Pipeline Overview

```
data/raw/  (9 source tables)
    ↓
01_data_loading.ipynb          ← YOU ARE HERE  (inspect sources)
    ↓
02_column_cleaning.ipynb       (drop constants, normalize columns, save cleaned tables)
    ↓
03_integration_and_features.ipynb  (merge, engineer features, encode)
    ↓
04_final_preparation.ipynb     (filter, validate, save final dataset)
    ↓
data/processed/algeria_export_opportunities_modeling_ready.csv
```

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

PALETTE = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B',
           '#44BBA4', '#E94F37', '#393E41', '#F5A623', '#7B2D8B']
sns.set_palette(PALETTE)
plt.rcParams['figure.figsize'] = (14, 4)

RAW_DIR = Path('../data/raw')

## 1. Load All Source Tables

In [ ]:
FILE_MAP = {
    'world_imports'     : '01_comtrade_world_imports_cleaned.csv',
    'algeria_production': '02_algeria_resources_fao_cleaned.csv',
    'algeria_inputs'    : '03_algeria_resources_fao_inputs_cleaned.csv',
    'algeria_land'      : '04_algeria_resources_fao_land_cleaned.csv',
    'algeria_prices'    : '05_algeria_resources_fao_prices_cleaned.csv',
    'algeria_worldbank' : '06_algeria_resources_worldbank_cleaned.csv',
    'unit_values'       : '09_comtrade_unit_values_cleaned.csv',
    'country_metadata'  : '11_country_metadata_cleaned.csv',
    'algeria_exports'   : '12_algeria_exports_comtrade_cleaned.csv',
}

dfs = {}
for name, fname in FILE_MAP.items():
    fpath = RAW_DIR / fname
    if fpath.exists():
        dfs[name] = pd.read_csv(fpath, low_memory=False)
        print(f"✅  {name:<22} {len(dfs[name]):>8,} rows  {len(dfs[name].columns):>3} cols")
    else:
        print(f"❌  {name:<22} FILE NOT FOUND — expected at {fpath}")

print(f"\nLoaded {len(dfs)} / {len(FILE_MAP)} tables")

## 2. Source Table Inventory

| Table | Source | What it contains | Key join fields |
|---|---|---|---|
| `world_imports` | UN Comtrade | Global import flows by country × product × year | `hs_code_6digit`, `importer_iso3`, `year` |
| `algeria_production` | FAOSTAT | Algeria's production volume, yield, area per crop per year | `hs_code_6digit`, `Year` |
| `algeria_inputs` | FAOSTAT | Algeria's fertilizer use and imports per year | `Year` |
| `algeria_land` | FAOSTAT | Algeria's agricultural land area and value per year | `Year` |
| `algeria_prices` | FAOSTAT | Algeria's producer prices per product per year | `Year` |
| `algeria_worldbank` | World Bank | Algeria's macroeconomic indicators per year | `year` |
| `unit_values` | UN Comtrade | World-average unit values (USD/kg) per product per year | `hs_code_6digit`, `year` |
| `country_metadata` | World Bank | Importer country macro indicators (GDP, population…) | `country_iso3` |
| `algeria_exports` | UN Comtrade | Algeria's actual exports by product × destination × year | `hs_code_6digit`, `partner_name`, `year` |

## 3. Column Inventory per Table

In [ ]:
def inspect(name: str, n_sample: int = 3) -> None:
    """Print dtype, null%, unique count, and sample values for each column."""
    df = dfs[name]
    print(f"\n{'='*65}")
    print(f"  TABLE: {name}   →   {df.shape[0]:,} rows  x  {df.shape[1]} cols")
    print(f"{'='*65}")
    for col in df.columns:
        dtype   = str(df[col].dtype)
        null_p  = df[col].isnull().mean() * 100
        n_uniq  = df[col].nunique(dropna=True)
        samples = df[col].dropna().unique()[:n_sample]
        print(f"  {col:<35} {dtype:<10} null={null_p:4.1f}%  uniq={n_uniq:>6}  → {samples}")

In [ ]:
inspect('world_imports')

In [ ]:
inspect('algeria_production')

In [ ]:
inspect('algeria_inputs')

In [ ]:
inspect('algeria_land')

In [ ]:
inspect('algeria_prices')

In [ ]:
inspect('algeria_worldbank')

In [ ]:
inspect('unit_values')

In [ ]:
inspect('country_metadata')

In [ ]:
inspect('algeria_exports')

## 4. Join Plan

Below is the join chain executed in `03_integration_and_features.ipynb`:

```
world_imports  (base table — one row per exporter × importer × product × year)
  LEFT JOIN unit_values        ON (hs_code_6digit, year)        — adds world avg price
  LEFT JOIN algeria_production ON (hs_code_6digit, year)        — adds Algeria production stats
  LEFT JOIN algeria_prices     ON (year)                        — adds Algeria producer price index
  LEFT JOIN algeria_inputs     ON (year)                        — adds fertilizer usage
  LEFT JOIN algeria_worldbank  ON (year)                        — adds Algeria macro context
  LEFT JOIN country_metadata   ON (importer_iso3)               — adds importer GDP, population, etc.
  LEFT JOIN algeria_exports    ON (hs_code_6digit, importer, year) — adds TARGET: Algeria export value
```

**Important row types in `world_imports`:**
- `exporter_name = 'World'`  → total market demand row (used for opportunity sizing)
- `exporter_name = <country>` → bilateral row (specific supplier to that importer)

Engineered features (`export_gap`, `market_share`, `price_competitiveness`) are **only meaningful on World rows**.

---
**Next:** `02_column_cleaning.ipynb` — drop redundant/constant columns, normalize units, save cleaned tables.